# iTunes Music Dataset – Story & Insights
## Assignment 2: Final Project

---

## The Story: What Makes a Track Successful on iTunes?

After a thorough EDA, three core themes emerged from this dataset:

1. **Genre dominance** — Pop and Bollywood together account for nearly half of all tracks, yet they behave very differently in terms of duration and pricing.
2. **Pricing is nearly uniform** — The vast majority of tracks cluster at premium pricing ($1.29), raising questions about what drives differentiation.
3. **Music production growth peaks around 2017** — The catalog shows a clear surge from 2014–2017, followed by a gradual decline that may reflect changing consumption patterns (streaming vs. download).

Each visualization below supports one chapter of this story.

---

In [4]:
pip install plotly 

  Obtaining dependency information for plotly from https://files.pythonhosted.org/packages/52/d2/c6e44dba74f17c6216ce1b56044a9b93a929f1c2d5bdaff892512b260f5e/plotly-6.6.0-py3-none-any.whl.metadata
  Obtaining dependency information for narwhals>=1.15.1 from https://files.pythonhosted.org/packages/3f/c3/06490e98393dcb4d6ce2bf331a39335375c300afaef526897881fbeae6ab/narwhals-2.18.1-py3-none-any.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 7.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 445.0/445.0 kB 8.6 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')


In [2]:
df = pd.read_csv('/Users/vahramdressler/Desktop/YSU/S2/Data Viz/Data-Viz/Data/itunes_music_dataset.csv')

df = df.dropna(subset=['artist_name', 'release_date'])
df['album_artist'] = df['album_artist'].fillna(df['artist_name'])
df['track_price'] = df['track_price'].fillna(df['track_price'].median())
df['collection_price'] = df['collection_price'].fillna(df['collection_price'].median())

median_price = df[df['track_price'] > 0]['track_price'].median()
df.loc[df['track_price'] < 0, 'track_price'] = median_price

# Remove single extreme-length outlier found in EDA
df = df.drop(df['track_time_millis'].idxmax()).reset_index(drop=True)

# Derived columns
df['duration_min']   = df['track_time_millis'] / 60_000
df['release_year']   = pd.to_datetime(df['release_date']).dt.year
df['price_tier']     = pd.cut(
    df['track_price'],
    bins=[-0.01, 0.70, 1.00, 1.30],
    labels=['Budget ($0.69)', 'Standard ($0.99)', 'Premium ($1.29)']
)

print(f'Dataset ready: {df.shape[0]:,} tracks | {df["genre"].nunique()} genres | {df["artist_name"].nunique()} artists')

Dataset ready: 4,864 tracks | 97 genres | 2327 artists


In [14]:
pip install --upgrade pip

  Obtaining dependency information for pip from https://files.pythonhosted.org/packages/de/f0/c81e05b613866b76d2d1066490adf1a3dbc4ee9d9c839961c3fc8a6997af/pip-26.0.1-py3-none-any.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 5.4 MB/s eta 0:00:00a 0:00:010m
  Attempting uninstall: pip
    Found existing installation: pip 23.2.1
    Uninstalling pip-23.2.1:
      Successfully uninstalled pip-23.2.1
Note: you may need to restart the kernel to use updated packages.


In [4]:
genre_counts = df['genre'].value_counts().head(15).reset_index()
genre_counts.columns = ['genre', 'count']
genre_counts['pct'] = (genre_counts['count'] / genre_counts['count'].sum() * 100).round(1)

fig1 = px.treemap(
    genre_counts,
    path=['genre'],
    values='count',
    color='count',
    color_continuous_scale='Teal',
    custom_data=['pct'],
    title='<b>Top 15 Genres by Track Count</b><br><sup>Size = number of tracks · Color intensity = dominance</sup>'
)
fig1.update_traces(
    texttemplate='<b>%{label}</b><br>%{value:,} tracks<br>%{customdata[0]:.1f}%',
    textfont_size=13
)
fig1.update_layout(height=500, coloraxis_showscale=False, margin=dict(t=80, l=10, r=10, b=10))
fig1.show()

print('\nInsight: Pop (22.7%) and Bollywood (23.0%) alone account for ~46% of all catalog tracks.')


Insight: Pop (22.7%) and Bollywood (23.0%) alone account for ~46% of all catalog tracks.



Knowing track counts is not enough — we also need to understand estimated revenue contribution. Since all tracks are individually priced, total revenue potential = number of tracks × average price. Bollywood leads in absolute potential, but niche genres like Rock and Hip-Hop/Rap punch above their weight with higher average prices.

In [10]:
genre_rev = df.groupby('genre').agg(
    num_tracks=('track_id', 'count'),
    avg_price=('track_price', 'mean'),
    total_revenue=('track_price', 'sum'),
    avg_duration=('duration_min', 'mean')
).reset_index()

genre_rev = genre_rev[genre_rev['num_tracks'] >= 50].sort_values('total_revenue', ascending=False)

fig2 = px.scatter(
    genre_rev,
    x='num_tracks',
    y='avg_price',
    size='total_revenue',
    color='total_revenue',
    color_continuous_scale='Viridis',
    text='genre',
    hover_data={'total_revenue': ':$.2f', 'avg_duration': ':.2f', 'num_tracks': True},
    title='<b>Genre Revenue Landscape</b>',
    labels={
        'num_tracks': 'Number of Tracks',
        'avg_price': 'Average Price ($)',
        'total_revenue': 'Total Revenue ($)'
    }
)
fig2.update_traces(textposition='top center', textfont_size=10)
fig2.update_layout(
    height=550,
    xaxis=dict(showgrid=True, gridcolor='#eee'),
    yaxis=dict(range=[1.05, 1.35], showgrid=True, gridcolor='#eee'),
    coloraxis_colorbar=dict(title='Revenue ($)'),
    plot_bgcolor='white'
)
fig2.show()

print('\nInsight: All western genras have fewer tracks than Bollywood but a higher average price,')
print('making it more efficient per track despite its smaller catalog size.')
print('However due to the amount of songs Bollywood comes second, behind the Pop.')


Insight: All western genras have fewer tracks than Bollywood but a higher average price,
making it more efficient per track despite its smaller catalog size.
However due to the amount of songs Bollywood comes second, behind the Pop.



The iTunes pricing model is highly concentrated. Over **78%** of tracks are priced at the premium tier ($1.29). This near-uniform structure means pricing alone is not a differentiator — instead, what matters is catalog depth and artist variety within each genre.

In [13]:
tier_counts = df['price_tier'].value_counts().reset_index()
tier_counts.columns = ['tier', 'count']
tier_counts['pct'] = (tier_counts['count'] / tier_counts['count'].sum() * 100).round(1)
tier_counts = tier_counts.sort_values('tier')

colors = ['#3d85c8', '#f6a623', '#e74c3c']

fig3 = go.Figure(go.Bar(
    x=tier_counts['tier'],
    y=tier_counts['count'],
    text=[f'{p:.1f}%' for p in tier_counts['pct']],
    textposition='outside',
    textfont=dict(size=15, color='black'),
    marker_color=colors,
    hovertemplate='<b>%{x}</b><br>Tracks: %{y:,}<extra></extra>'
))

fig3.update_layout(
    title='<b>Price Tier Distribution</b>',
    xaxis_title='Price Tier',
    yaxis_title='Number of Tracks',
    plot_bgcolor='white',
    yaxis=dict(showgrid=True, gridcolor='#eee'),
    height=800,
    margin=dict(t=100)
)
fig3.show()

print('\nInsight: 78.9% of tracks are priced at $1.29 (Premium). Pricing is not a competitive lever on iTunes.')


Insight: 78.9% of tracks are priced at $1.29 (Premium). Pricing is not a competitive lever on iTunes.



Track duration varies meaningfully across genres. Bollywood and Soundtracks run noticeably longer (~4.5 min average), reflecting their cinematic nature. Country and Indian Pop are tighter compositions. This has real implications: longer tracks may feel like better value to consumers but also require higher production investment.

In [29]:
top8_genres = df['genre'].value_counts().head(8).index.tolist()
df_top8 = df[df['genre'].isin(top8_genres)].copy()

# Order by median duration
order = df_top8.groupby('genre')['duration_min'].median().sort_values().index.tolist()

fig4 = go.Figure()

palette = px.colors.qualitative.Safe
for i, genre in enumerate(order):
    data = df_top8[df_top8['genre'] == genre]['duration_min']
    fig4.add_trace(go.Box(
        y=data,
        name=genre,
        boxmean=True,
        marker_color=palette[i % len(palette)],
        line=dict(color='#333333', width=2),   # dark line makes median pop
        fillcolor=palette[i % len(palette)],
        opacity=0.7,
        hoverinfo='y+name'
    ))

fig4.update_layout(
    title='<b>Track Duration Distribution by Genre</b><br><sup>Top 8 genres · Ordered by median duration · Box shows IQR · Dashed line shows mean</sup>',
    yaxis_title='Duration (minutes)',
    xaxis_title='Genre',
    plot_bgcolor='white',
    yaxis=dict(showgrid=True, gridcolor='#eee', range=[0, 12]),
    height=520,
    showlegend=False
)
fig4.show()

print('\nInsight: Bollywood (~4.5 min) and Soundtrack (~4.5 min) tracks run ~25% longer than')
print('Country tracks (~3.5 min), reflecting their cinematic production style.')


Insight: Bollywood (~4.5 min) and Soundtrack (~4.5 min) tracks run ~25% longer than
Country tracks (~3.5 min), reflecting their cinematic production style.



Looking at release year trends reveals a compelling arc: the iTunes catalog grew steadily from 2010, peaked around **2016–2017**, and has been declining since. This mirrors the broader industry shift toward streaming (Spotify, Apple Music), where consumers rent music rather than purchase individual tracks.

In [30]:
year_df = df[(df['release_year'] >= 2005) & (df['release_year'] <= 2025)]
year_counts = year_df.groupby('release_year').size().reset_index(name='tracks')

df['genre_group'] = df['genre'].apply(
    lambda g: 'Bollywood / Indian' if g in ['Bollywood', 'Indian Pop', 'Punjabi Pop', 'Telugu']
    else ('Western' if g in ['Pop', 'Rock', 'Hip-Hop/Rap', 'R&B/Soul', 'Country', 'Alternative', 'Dance', 'Electronic']
    else 'Other')
)

year_genre = (
    df[(df['release_year'] >= 2005) & (df['release_year'] <= 2025)]
    .groupby(['release_year', 'genre_group'])
    .size()
    .reset_index(name='tracks')
)

fig5 = px.area(
    year_genre,
    x='release_year',
    y='tracks',
    color='genre_group',
    color_discrete_map={
        'Western': '#3d85c8',
        'Bollywood / Indian': '#f6a623',
        'Other': '#a8d8a8'
    },
    title='<b>Tracks Released Per Year (2005–2025)</b>',
    labels={'release_year': 'Year', 'tracks': 'Number of Tracks', 'genre_group': 'Genre Group'}
)

fig5.add_annotation(
    x=2017, y=332, text='<b>Peak: 2017</b>',
    showarrow=True, arrowhead=2, ax=40, ay=-40,
    font=dict(color='#c0392b', size=13),
    arrowcolor='#c0392b'
)

fig5.update_layout(
    height=480,
    plot_bgcolor='white',
    xaxis=dict(showgrid=False, dtick=2),
    yaxis=dict(showgrid=True, gridcolor='#eee'),
    legend=dict(title='Genre Group', orientation='h', y=-0.15)
)
fig5.show()

print('\nInsight: Catalog releases peaked in 2016-2017 and have significantly declined  by 2024,')
print('consistent with the shift from individual track purchases to streaming subscriptions.')


Insight: Catalog releases peaked in 2016-2017 and have significantly declined  by 2024,
consistent with the shift from individual track purchases to streaming subscriptions.


In [31]:
top_artists = df['artist_name'].value_counts().head(12).reset_index()
top_artists.columns = ['artist', 'count']

def artist_group(artist):
    sa = ['Diljit Dosanjh', 'Sidhu Moose Wala', 'Kishore Kumar', 'A.R. Rahman',
          'Arijit Singh', 'Lata Mangeshkar', 'Sonu Nigam']
    return 'South Asian' if artist in sa else 'Western'

top_artists['group'] = top_artists['artist'].apply(artist_group)
top_artists = top_artists.sort_values('count')

fig6 = px.bar(
    top_artists,
    x='count',
    y='artist',
    color='group',
    orientation='h',
    color_discrete_map={'Western': '#3d85c8', 'South Asian': '#f6a623'},
    text='count',
    title='<b>Top 12 Artists by Track Count</b>',
    labels={'count': 'Number of Tracks', 'artist': '', 'group': 'Artist Origin'}
)
fig6.update_traces(textposition='outside')
fig6.update_layout(
    height=520,
    plot_bgcolor='white',
    xaxis=dict(showgrid=True, gridcolor='#eee'),
    legend=dict(orientation='h', y=-0.12)
)
fig6.show()

print('\nInsight: Taylor Swift (105) leads, followed by The Beatles (96) and Michael Jackson (89).')
print('2 of the top 12 are South Asian artists, confirming the Bollywood catalog depth seen in Chapter 1.')


Insight: Taylor Swift (105) leads, followed by The Beatles (96) and Michael Jackson (89).
2 of the top 12 are South Asian artists, confirming the Bollywood catalog depth seen in Chapter 1.



A natural question: does track length influence its price? On iTunes, the answer is largely **no** — pricing is pre-set by tiers, not by duration. However, looking at genre clusters reveals that longer tracks (Bollywood, Soundtrack) tend to congregate in the standard pricing band, while premium pricing is more uniformly distributed.

In [32]:
df_plot = df[df['genre'].isin(top8_genres) & (df['duration_min'] <= 10)].copy()

fig7 = px.scatter(
    df_plot,
    x='duration_min',
    y='track_price',
    color='genre',
    opacity=0.45,
    color_discrete_sequence=px.colors.qualitative.Safe,
    title='<b>Track Duration vs. Price by Genre</b>',
    labels={'duration_min': 'Track Duration (minutes)', 'track_price': 'Price ($)', 'genre': 'Genre'},
    hover_data=['artist_name', 'track_name']
)

for price, label, color in [(0.69, '$0.69 Budget', '#3d85c8'), (0.99, '$0.99 Standard', '#f6a623'), (1.29, '$1.29 Premium', '#e74c3c')]:
    fig7.add_hline(y=price, line_dash='dot', line_color=color, annotation_text=label, annotation_position='right')

fig7.update_layout(
    height=520,
    plot_bgcolor='white',
    xaxis=dict(showgrid=True, gridcolor='#eee'),
    yaxis=dict(showgrid=False, tickvals=[0.69, 0.99, 1.29]),
    legend=dict(orientation='h', y=-0.18)
)
fig7.show()

print('\nInsight: Price does NOT correlate with duration. Tracks are priced by tier,')
print('regardless of whether they are 2 minutes or 8 minutes long.')


Insight: Price does NOT correlate with duration. Tracks are priced by tier,
regardless of whether they are 2 minutes or 8 minutes long.


---

## Summary:

| # | Insight 
|---|---------
| 1 | **Pop & Bollywood rule** — together 40% of all tracks 
| 2 | **Premium pricing dominates** — 79% of tracks cost $1.29 
| 3 | **Catalog growth peaked in 2017** — streaming is winning 
| 4 | **Duration ≠ Price** — length doesn't buy a better tier 

